In [1]:
import numpy as np
import pandas as pd

df = pd.read_json("resumes_dataset.jsonl", lines=True)
df.head()

,ResumeID,Category,Name,Email,Phone,Location,Summary,Skills,Experience,Education,Text,Source
0,REAL_0001,Java Developer,Chad Griffin,contact@email.com,94105 555 4321000 10 ...,"City, State",jessica claire montgomery street san francisco...,"Python, SQL, Git, Linux",jessica claire montgomery street san francisco...,Computer Science degree,jessica claire montgomery street san francisco...,ResumeAtlas
1,REAL_0002,Java Developer,Melinda Thomas,contact@email.com,17994568777 2017 2018 20152016 3 ...,"City, State",jared arthur maica java developer 17994568777 ...,"Python, SQL, Git, Linux",jared arthur maica java developer 17994568777 ...,Computer Science degree,jared arthur maica java developer 17994568777 ...,ResumeAtlas
2,REAL_0003,Java Developer,Shannon Mccarthy,contact@email.com,9 555 4321000 94105 8 ...,"City, State",jessica claire 9 resumesampleexamplecom 555 43...,"Python, SQL, Git, Linux",jessica claire 9 resumesampleexamplecom 555 43...,Computer Science degree,jessica claire 9 resumesampleexamplecom 555 43...,ResumeAtlas
3,REAL_0004,Java Developer,Christine Kelley,contact@email.com,9 555 4321000 94105 5 ...,"City, State",jessica claire 9 resumesampleexamplecom 555 43...,"Python, SQL, Git, Linux",jessica claire 9 resumesampleexamplecom 555 43...,Computer Science degree,jessica claire 9 resumesampleexamplecom 555 43...,ResumeAtlas
4,REAL_0005,Java Developer,Karen Holt,contact@email.com,100 10 4321000 ...,"City, State",jessica claire 100 montgomery st 10th floor xx...,"Python, SQL, Git, Linux",jessica claire 100 montgomery st 10th floor xx...,Computer Science degree,jessica claire 100 montgomery st 10th floor xx...,ResumeAtlas


In [2]:
df.columns

Index(['ResumeID', 'Category', 'Name', 'Email', 'Phone', 'Location', 'Summary',
       'Skills', 'Experience', 'Education', 'Text', 'Source'],
      dtype='object')

In [3]:
needed_columns = [
    'Category',
    'Summary',
    'Skills',
    'Experience',
    'Education',
    'Text'
]
df = df[[column for column in needed_columns if column in df.columns]]

In [4]:
df.head()

,Category,Summary,Skills,Experience,Education,Text
0,Java Developer,jessica claire montgomery street san francisco...,"Python, SQL, Git, Linux",jessica claire montgomery street san francisco...,Computer Science degree,jessica claire montgomery street san francisco...
1,Java Developer,jared arthur maica java developer 17994568777 ...,"Python, SQL, Git, Linux",jared arthur maica java developer 17994568777 ...,Computer Science degree,jared arthur maica java developer 17994568777 ...
2,Java Developer,jessica claire 9 resumesampleexamplecom 555 43...,"Python, SQL, Git, Linux",jessica claire 9 resumesampleexamplecom 555 43...,Computer Science degree,jessica claire 9 resumesampleexamplecom 555 43...
3,Java Developer,jessica claire 9 resumesampleexamplecom 555 43...,"Python, SQL, Git, Linux",jessica claire 9 resumesampleexamplecom 555 43...,Computer Science degree,jessica claire 9 resumesampleexamplecom 555 43...
4,Java Developer,jessica claire 100 montgomery st 10th floor xx...,"Python, SQL, Git, Linux",jessica claire 100 montgomery st 10th floor xx...,Computer Science degree,jessica claire 100 montgomery st 10th floor xx...


In [5]:
def merge_resume_fields(row):
    fields = [
        row.get('Summary', ''),
        row.get('Skills', ''),
        row.get('Experience', ''),
        row.get('Education', ''),
        row.get('Text', '')
    ]
    return ' '.join(str(f) for f in fields if pd.notna(f))

df['Resume'] = df.apply(merge_resume_fields, axis=1)


In [6]:
df = df[['Category', 'Resume']]
df.head()

,Category,Resume
0,Java Developer,jessica claire montgomery street san francisco...
1,Java Developer,jared arthur maica java developer 17994568777 ...
2,Java Developer,jessica claire 9 resumesampleexamplecom 555 43...
3,Java Developer,jessica claire 9 resumesampleexamplecom 555 43...
4,Java Developer,jessica claire 100 montgomery st 10th floor xx...


In [7]:
df['Category'].value_counts()

Category
Java Developer               200
Python Developer             200
Data Science                 200
DevOps                       180
SQL Developer                180
Database                     150
Testing                      150
Web Designing                150
React Developer              150
Business Analyst             150
DotNet Developer             140
Software Developer           134
Network Security Engineer    120
ETL Developer                120
Full Stack Developer         102
SAP Developer                100
Digital Media                100
Cloud Engineer                92
Machine Learning Engineer     81
Frontend Developer            76
Backend Developer             76
AI Engineer                   71
Cybersecurity Analyst         66
QA Engineer                   61
Database Administrator        56
UI/UX Designer                51
Blockchain                    47
Site Reliability Engineer     46
Mobile Developer              46
System Administrator          40
T

In [8]:
df['Resume'] = df['Resume'].str.lower()
df['Resume'] = df['Resume'].str.strip()
df['Resume'] = df['Resume'].str.replace(r'\s+', ' ', regex=True)  #remove extra spaces

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=5000,
        ngram_range=(1, 2),
        max_df=0.85,
        min_df=2
    )),
    ('clf', LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="lbfgs",
        multi_class="auto"
    ))
])

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X = df['Resume']
le = LabelEncoder()
y = df['Category']
y = le.fit_transform(y)
y

array([17, 17, 17, ..., 32, 32, 32], shape=(3500,))

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model_pipeline.fit(X_train, y_train)

c:\Users\amogh\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,steps,"[('tfidf', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [12]:
from sklearn.metrics import accuracy_score
y_test_predicted = model_pipeline.predict(X_test)
print(accuracy_score(y_test, y_test_predicted))

0.8885714285714286


In [13]:
def guess_top5_jobs(resume):
    pred = model_pipeline.predict_proba(resume)
    pred = pred.ravel()
    pred_confidence = np.sort(pred)
    # Use argsort to get indices that would sort the array in ascending order
    sorted_indices_asc = np.argsort(pred)
    # To get the top (largest) values, reverse the array of indices and slice the first 'num_elements'
    top_pred_confidences = pred_confidence[::-1][:5]
    top_indices = sorted_indices_asc[::-1][:5]
    return list(zip(le.inverse_transform(top_indices), top_pred_confidences))

In [14]:
sample_resume_1 = ["""PROFESSIONAL SUMMARY
Detail-oriented Test Engineer with 5+ years of experience in Software Quality Assurance, specializing in automated testing frameworks, API testing, and performance analysis. Proven track record of increasing testing efficiency by 70% and reducing post-release defects by 30% through robust automation and Agile methodologies. 
TECHNICAL SKILLS
Testing Tools: Selenium WebDriver, Cucumber, JUnit, TestNG, Postman
Languages: Java, Python, SQL
CI/CD & DevOps: Jenkins, Azure Pipelines, Git
Test Management: Jira, Zephyr, TestRail
Methodologies: Agile/Scrum, Functional Testing, Regression Testing, API Testing 
PROFESSIONAL EXPERIENCE
[Company Name] | [Location]
QA Test Engineer | [Start Date] – [Present] 
Reduced manual testing effort by 80% by implementing automated regression testing scripts using Selenium WebDriver and Java.
Collaborated with developers in Agile sprints to resolve 25+ critical defects, ensuring a 98% test case pass rate.
Designed and executed API tests using Postman, reducing API-related defects by 20%.
Integrated automated test suites into Jenkins CI/CD pipeline, decreasing deployment time by 35%. 
[Previous Company Name] | [Location]
Junior Test Engineer | [Start Date] – [End Date] 
Executed 500+ manual test cases per release, identifying and reporting 50+ defects.
Developed comprehensive test plans and scenarios that increased test coverage from 0% to 40%.
Utilized Jira for tracking bugs, leading to a 30% reduction in bug-reporting errors. 
EDUCATION
Bachelor of Science in Computer Science
[University Name], [City, State]
CERTIFICATIONS
ISTQB Certified Tester, Foundation Level (CTFL) """]

top5_jobs = guess_top5_jobs(sample_resume_1)
top5_jobs

[('Testing', np.float64(0.6105274532755804)),
 ('DevOps', np.float64(0.03805361969553665)),
 ('Java Developer', np.float64(0.028552084230660503)),
 ('ETL Developer', np.float64(0.027717766567103005)),
 ('Python Developer', np.float64(0.02736773376666809))]

In [15]:
sample_resume_2 = ["""Name: Rahul Mehta

Professional Summary:
Highly analytical professional with experience spanning data-driven decision making, software development,
process optimization, and stakeholder management. Adept at transforming business requirements into
technical solutions using programming, automation tools, and statistical techniques.

Skills:
Programming Languages: Python, Java, SQL, JavaScript
Data & Analytics: Pandas, NumPy, Scikit-learn, Power BI, Excel, Data Visualization, Statistical Analysis
Web & Automation: HTML, CSS, REST APIs, Flask, Selenium
Business & Operations: Requirement Gathering, Process Improvement, KPI Tracking, Risk Analysis
Tools: Git, Docker, JIRA, SAP MM Basics, Linux
Soft Skills: Client Communication, Team Leadership, Documentation, Presentation

Professional Experience:

Business Data Analyst | XYZ Solutions | 2022 – Present
- Analyzed large datasets to identify trends and insights for improving operational efficiency.
- Built predictive models using Python and machine learning techniques to forecast sales and demand.
- Worked closely with stakeholders to translate business requirements into technical specifications.
- Automated reporting pipelines and dashboards using Python and Power BI.
- Collaborated with DevOps teams to deploy analytics solutions.

Software Engineer | ABC Tech | 2020 – 2022
- Developed backend services using Java and Python for enterprise applications.
- Designed RESTful APIs and integrated databases using SQL.
- Implemented automation scripts for testing and monitoring.
- Participated in Agile ceremonies and coordinated with cross-functional teams.

Academic Projects:
- Resume Classification System using NLP, TF-IDF, and Logistic Regression
- Sales Forecasting Model using Machine Learning and Time Series Analysis
- Web-based Dashboard for Business Metrics using Flask and JavaScript

Certifications:
- Google Data Analytics Certificate
- SAP MM Overview Training
- Python for Machine Learning

Interests:
Data Science, Business Strategy, Automation, Process Optimization
"""]

top5_jobs = guess_top5_jobs(sample_resume_2)
top5_jobs

[('Python Developer', np.float64(0.1685514184652375)),
 ('Data Science', np.float64(0.15181261650456315)),
 ('Business Analyst', np.float64(0.08916975724496246)),
 ('React Developer', np.float64(0.05904834670803541)),
 ('ETL Developer', np.float64(0.05527056728472668))]

In [16]:
sample_resume_3 = ["""
Name: Ananya Verma

Profile Summary:
Problem-solving professional with hands-on experience in building data-driven solutions,
optimizing information flows, and supporting decision-making across technical and non-technical teams.
Comfortable working with structured and semi-structured data to deliver reliable insights.

Core Competencies:
• Scripting & Programming: Python, SQL
• Data Handling: Data cleaning, transformation, validation
• Analysis: Exploratory analysis, trend identification, reporting
• Systems & Tools: Linux, Git, cloud-based data platforms
• Visualization & Reporting: Dashboards, automated reports
• Collaboration: Cross-functional coordination, documentation

Work Experience:

Data Solutions Associate | InsightEdge Pvt. Ltd. | 2021 – Present
- Prepared and transformed raw datasets from multiple sources into analysis-ready formats.
- Developed reusable scripts to automate routine data processing tasks.
- Designed reports to support business and operational decisions.
- Worked with engineering teams to ensure data consistency and reliability.
- Assisted analysts and managers with ad-hoc data requests.

Junior Systems Analyst | CoreLogic Systems | 2019 – 2021
- Supported internal teams by maintaining structured data repositories.
- Wrote SQL queries to extract and summarize information for stakeholders.
- Helped monitor system performance and data pipelines.
- Documented processes and standard operating procedures.

Projects:
- Automated data preparation workflow using Python and SQL
- Operational reporting system for weekly performance tracking
- Data quality monitoring scripts

Education:
Bachelor of Technology in Computer Science

Interests:
Process automation, data reliability, scalable systems
"""]
top5_jobs = guess_top5_jobs(sample_resume_3)
top5_jobs

[('Python Developer', np.float64(0.14998305088607608)),
 ('Data Science', np.float64(0.13330028598309002)),
 ('ETL Developer', np.float64(0.129905927008553)),
 ('SQL Developer', np.float64(0.06806128301929909)),
 ('Database', np.float64(0.065904751831649))]

In [17]:
sample_resume_4 = ["""
Name: Suresh Kumar

Professional Summary:
Detail-oriented engineering professional with strong experience in designing, executing,
and supervising infrastructure and system-level projects. Specialized in planning,
implementation, compliance, and cross-team coordination for large-scale technical operations.

Core Skills:
• Domain Expertise: Structural design, load analysis, site supervision
• Engineering Tools: AutoCAD, STAAD.Pro, ETABS
• Project Management: Scheduling, cost estimation, resource planning
• Compliance & Safety: Quality assurance, safety standards, technical documentation
• Materials & Processes: Concrete technology, steel structures, foundation systems
• Software Knowledge: MS Project, Excel, basic Python (scripting familiarity)

Work Experience:

Site Engineer | Horizon Infrastructure Ltd. | 2020 – Present
- Supervised on-site construction activities and ensured adherence to approved designs.
- Coordinated with contractors, vendors, and consultants to maintain project timelines.
- Performed quality checks and compliance inspections as per safety regulations.
- Prepared technical reports, progress documentation, and material requirement plans.
- Assisted in resolving design and execution challenges on-site.

Design Engineer | BuildTech Consultants | 2018 – 2020
- Created structural drawings and layout plans using AutoCAD and ETABS.
- Conducted load calculations and feasibility studies for commercial buildings.
- Supported senior engineers in design validation and client presentations.
- Maintained documentation for regulatory approvals.

Education:
Bachelor of Technology in Civil Engineering

Certifications:
- AutoCAD Professional Certification
- Construction Safety & Quality Management

Additional Information:
- Familiar with Python for basic calculations and automation
- Strong understanding of field operations and technical coordination

Interests:
Infrastructure development, sustainable construction, project execution
"""]
top5_jobs = guess_top5_jobs(sample_resume_4)
top5_jobs

[('Testing', np.float64(0.1300777813329916)),
 ('Web Designing', np.float64(0.05870582551490262)),
 ('React Developer', np.float64(0.0581946631520082)),
 ('DevOps', np.float64(0.05531884460255506)),
 ('Database', np.float64(0.05506735904042675))]

In [18]:
sample_resume_5 = ["""Name: Rohan Malhotra

Professional Summary:
IT infrastructure professional with hands-on experience in managing enterprise systems,
networks, and cloud environments. Strong background in system administration, monitoring,
incident response, and ensuring high availability of IT services.

Technical Skills:
• Operating Systems: Windows Server, Linux (Ubuntu, CentOS)
• Networking: TCP/IP, DNS, DHCP, VLANs, VPNs, Firewalls
• Cloud Platforms: AWS (EC2, S3, IAM), Azure Basics
• Virtualization: VMware, VirtualBox
• Monitoring & Support: Nagios, Zabbix, ServiceNow
• Security: Patch management, access control, basic SIEM concepts
• Scripting & Tools: Bash scripting, PowerShell, basic Python
• Databases: MySQL administration, backup & recovery
• DevOps Exposure: Docker fundamentals, CI/CD concepts

Professional Experience:

System Administrator | NetCore IT Services | 2021 – Present
- Managed and maintained Linux and Windows servers in production environments.
- Configured user access, permissions, and security policies.
- Monitored system health and responded to incidents to ensure uptime.
- Performed backups, recovery testing, and patch management.
- Worked with networking teams on firewall rules and VPN configurations.

IT Support Engineer | ByteWave Solutions | 2019 – 2021
- Provided Level 2 technical support for internal users and applications.
- Diagnosed hardware, software, and network-related issues.
- Maintained IT documentation and asset inventory.
- Assisted in cloud resource provisioning and cost monitoring.

Certifications:
- AWS Certified Cloud Practitioner
- Linux System Administration Fundamentals

Education:
Bachelor of Technology in Information Technology

Additional Information:
- Familiar with Python for automation scripts
- Strong troubleshooting and operational mindset

Interests:
Cloud infrastructure, system reliability, IT security
"""]
top5_jobs = guess_top5_jobs(sample_resume_5)
top5_jobs


[('DevOps', np.float64(0.23316025916681377)),
 ('Network Security Engineer', np.float64(0.1996782803311435)),
 ('Database', np.float64(0.07506490569258413)),
 ('Testing', np.float64(0.046990280102152816)),
 ('Python Developer', np.float64(0.04554710810028468))]

In [19]:
sample_resume_6 = ["""Experienced professional with a background in developing interactive web applications and analyzing data to support business decisions. Skilled in translating data insights into user-friendly interfaces and actionable reports. Strong understanding of both frontend development and analytical problem-solving.

Hands-on experience building responsive web applications using modern JavaScript frameworks, implementing clean UI components, and integrating backend APIs. Comfortable working with HTML, CSS, and JavaScript to create accessible, mobile-friendly user experiences. Experience collaborating with designers and backend engineers to deliver end-to-end web solutions.

Proficient in analyzing structured datasets using Python and SQL to identify trends, patterns, and performance metrics. Regularly performs data cleaning, exploratory analysis, and visualization to support reporting and decision-making. Experienced in creating dashboards and summaries that communicate insights clearly to technical and non-technical stakeholders.

Worked on projects that combine web development with data analysis, such as embedding analytical charts and reports into web applications. Experience handling user interaction data, monitoring usage metrics, and improving application features based on analytical findings.

Strong problem-solving skills with the ability to balance technical implementation and analytical reasoning. Interested in roles that blend web development and data analysis to create data-informed digital products."""]
top5_jobs = guess_top5_jobs(sample_resume_6)
top5_jobs

[('Python Developer', np.float64(0.1625325473568464)),
 ('Web Designing', np.float64(0.14054314988877878)),
 ('React Developer', np.float64(0.10860434473672757)),
 ('Data Science', np.float64(0.06840876808180922)),
 ('DotNet Developer', np.float64(0.056202985060557875))]

In [20]:
import joblib
joblib.dump(model_pipeline, "resume_model_v1.pkl")
joblib.dump(le, "label_encoder_v1.pkl")

['label_encoder_v1.pkl']